# 第18章 代码教学：GUI Agent 与工具调用（ReAct）

> 目标：
> 1) 用最小 GUI 状态机构建 Agent
> 2) 用 ReAct 结构组织“思考 -> 行动 -> 观察”
> 3) 在工具调用层加入参数约束与安全拦截


## 0. 环境准备（标准库即可）


In [1]:
import json, re
from dataclasses import dataclass, field
from datetime import datetime
from typing import Dict, List, Any
print('ready')


ready


## 1. 定义工具与安全约束

- 允许的应用白名单
- 邮件域名白名单
- 文本长度限制
- 时间格式校验


In [2]:
ALLOWED_APPS = {'desktop','mail','calendar','browser'}
ALLOWED_DOMAINS = {'company.com','partner.com'}
MAX_TEXT_LEN = 400

def check_email(addr):
    m = re.match(r'^[A-Za-z0-9._%+-]+@([A-Za-z0-9.-]+)$', addr)
    if not m: return False, 'invalid email format'
    d = m.group(1).lower()
    if d not in ALLOWED_DOMAINS: return False, f'domain not allowed: {d}'
    return True, 'ok'

def check_dt(s):
    try: datetime.strptime(s, '%Y-%m-%d %H:%M'); return True, 'ok'
    except Exception: return False, 'datetime must be YYYY-MM-DD HH:MM' 


## 2. GUI 状态（教学版）


In [3]:
@dataclass
class GUIState:
    current_app: str = 'desktop'
    fields: Dict[str, str] = field(default_factory=dict)
    clicks: List[str] = field(default_factory=list)
    events: List[Dict[str, Any]] = field(default_factory=list)
    sent_emails: List[Dict[str, str]] = field(default_factory=list)

state = GUIState()
state


GUIState(current_app='desktop', fields={}, clicks=[], events=[], sent_emails=[])

## 3. 工具执行器（含安全检查）


In [4]:
def open_app(args, s):
    app = args.get('app','')
    if app not in ALLOWED_APPS: return False, f'blocked app: {app}'
    s.current_app = app; return True, f'opened {app}'

def click(args, s):
    sel = args.get('selector','')
    if not sel: return False, 'selector required'
    s.clicks.append(sel); return True, f'clicked {sel}'

def type_text(args, s):
    field = args.get('field',''); text = args.get('text','')
    if not field: return False, 'field required'
    if len(text) > MAX_TEXT_LEN: return False, 'text too long'
    s.fields[field] = text; return True, f'typed {field}'

def create_event(args, s):
    if s.current_app != 'calendar': return False, 'calendar app required'
    ok, msg = check_dt(args.get('datetime',''))
    if not ok: return False, msg
    for a in args.get('attendees',[]):
        ok, msg = check_email(a)
        if not ok: return False, msg
    s.events.append({'title': args.get('title',''), 'datetime': args.get('datetime',''), 'attendees': args.get('attendees',[])})
    return True, 'event created'

def send_email(args, s):
    if s.current_app != 'mail': return False, 'mail app required'
    ok, msg = check_email(args.get('to',''))
    if not ok: return False, msg
    if len(args.get('body','')) > MAX_TEXT_LEN: return False, 'body too long'
    s.sent_emails.append({'to': args.get('to',''), 'subject': args.get('subject',''), 'body': args.get('body','')})
    return True, 'email sent'

TOOLS = {'open_app': open_app, 'click': click, 'type_text': type_text, 'create_event': create_event, 'send_email': send_email}


## 4. Action 解析与执行


In [5]:
def parse_action(text):
    try: obj = json.loads(text)
    except Exception: return False, 'invalid json', None
    if 'tool' not in obj: return False, 'missing tool', None
    if 'args' not in obj: obj['args'] = {}
    return True, 'ok', obj

def exec_action(obj, s):
    tool = obj['tool']; args = obj.get('args', {})
    if tool == 'finish': return True, 'finished'
    if tool not in TOOLS: return False, f'unknown tool: {tool}'
    return TOOLS[tool](args, s)


## 5. ReAct 规划器（教学版）

为了可复现，这里用规则模拟“模型决策输出”。


In [6]:
def plan(task, s, step):
    t = task.lower()
    if 'meeting' in t or '会议' in t:
        if step == 0: return {'tool':'open_app','args':{'app':'calendar'}}
        if step == 1: return {'tool':'create_event','args':{'title':'Project Sync','datetime':'2026-02-08 10:30','attendees':['alice@company.com','bob@partner.com']}}
        return {'tool':'finish','args':{}}
    if 'email' in t or '邮件' in t:
        if step == 0: return {'tool':'open_app','args':{'app':'mail'}}
        if step == 1: return {'tool':'send_email','args':{'to':'alice@company.com','subject':'Status Update','body':'Training pipeline is complete. Next step is evaluation.'}}
        return {'tool':'finish','args':{}}
    return {'tool':'finish','args':{}}


## 6. Agent 主循环（Thought -> Action -> Observation）


In [7]:
def run_agent(task, max_steps=6):
    s = GUIState(); trace = []
    for step in range(max_steps):
        thought = f'Step {step}: reason with app={s.current_app}'
        act = plan(task, s, step)
        txt = json.dumps(act, ensure_ascii=False)
        ok, msg, obj = parse_action(txt)
        if not ok:
            trace.append({'step': step, 'thought': thought, 'action': txt, 'obs': msg}); break
        ok2, obs = exec_action(obj, s)
        trace.append({'step': step, 'thought': thought, 'action': txt, 'obs': obs})
        if obj['tool'] == 'finish' or not ok2: break
    return trace, s


## 7. Demo A：创建会议


In [8]:
trace, s = run_agent('Please create a meeting with Alice and Bob tomorrow morning.')
for x in trace:
    print(f"[{x['step']}]", x['thought'])
    print(' action=', x['action'])
    print(' obs=', x['obs'])
    print('-'*60)
print('events=', s.events)


[0] Step 0: reason with app=desktop
 action= {"tool": "open_app", "args": {"app": "calendar"}}
 obs= opened calendar
------------------------------------------------------------
[1] Step 1: reason with app=calendar
 action= {"tool": "create_event", "args": {"title": "Project Sync", "datetime": "2026-02-08 10:30", "attendees": ["alice@company.com", "bob@partner.com"]}}
 obs= event created
------------------------------------------------------------
[2] Step 2: reason with app=calendar
 action= {"tool": "finish", "args": {}}
 obs= finished
------------------------------------------------------------
events= [{'title': 'Project Sync', 'datetime': '2026-02-08 10:30', 'attendees': ['alice@company.com', 'bob@partner.com']}]


## 8. Demo B：发送邮件


In [9]:
trace2, s2 = run_agent('Send an email to Alice about project status.')
for x in trace2:
    print(f"[{x['step']}]", x['thought'])
    print(' action=', x['action'])
    print(' obs=', x['obs'])
    print('-'*60)
print('emails=', s2.sent_emails)


[0] Step 0: reason with app=desktop
 action= {"tool": "open_app", "args": {"app": "mail"}}
 obs= opened mail
------------------------------------------------------------
[1] Step 1: reason with app=mail
 action= {"tool": "send_email", "args": {"to": "alice@company.com", "subject": "Status Update", "body": "Training pipeline is complete. Next step is evaluation."}}
 obs= email sent
------------------------------------------------------------
[2] Step 2: reason with app=mail
 action= {"tool": "finish", "args": {}}
 obs= finished
------------------------------------------------------------
emails= [{'to': 'alice@company.com', 'subject': 'Status Update', 'body': 'Training pipeline is complete. Next step is evaluation.'}]


## 9. 安全拦截演示（越权/注入）


In [10]:
atk = GUIState(current_app='mail')
mal = [
    {'tool':'send_email','args':{'to':'attacker@evil.com','subject':'x','body':'steal'}},
    {'tool':'create_event','args':{'title':'Hacked','datetime':'next week','attendees':['alice@company.com']}},
]
for a in mal:
    ok, obs = exec_action(a, atk)
    print('action=', a)
    print('ok=', ok, 'obs=', obs)
    print('-'*60)


action= {'tool': 'send_email', 'args': {'to': 'attacker@evil.com', 'subject': 'x', 'body': 'steal'}}
ok= False obs= domain not allowed: evil.com
------------------------------------------------------------
action= {'tool': 'create_event', 'args': {'title': 'Hacked', 'datetime': 'next week', 'attendees': ['alice@company.com']}}
ok= False obs= calendar app required
------------------------------------------------------------


## 10. 练习
1) 增加 search_docs 工具；2) 加 action budget；3) 写审计日志与合规检查。
